# Visualization — matplotlib and seaborn

Notebook 04 left you with clean, reshaped DataFrames. Plots are how you actually look at them.

This notebook covers two libraries side by side.

1. **`matplotlib`** — the low-level plotting engine. Gives you full control over the figure, the axes, the ticks, the colors. The mental model — `Figure` and `Axes` — carries over to every plot you will ever draw.
2. **`seaborn`** — sits on top of matplotlib and gives you statistical plots in one line. Distributions, relationships, categorical aggregates, correlation matrices.

You will end up using both. matplotlib for the structure of the figure; seaborn for the actual plot inside each axes.


## Two libraries, one mental model

`matplotlib` is the canvas. `seaborn` is the paint.

Every seaborn function calls matplotlib under the hood. When you write `sns.histplot(data=df, x="age")`, seaborn creates (or accepts) a matplotlib `Axes` object, draws onto it, and returns it. That means three useful things hold simultaneously.

- You can customize any seaborn plot using matplotlib methods on the returned `ax`.
- You can put multiple seaborn plots in one figure by passing each one its own `ax=...`.
- You never have to choose one library over the other. Use both at once.


## The Figure / Axes model

This is the single most important concept in matplotlib.

- A **`Figure`** is the whole image — the rectangle the file gets saved as.
- An **`Axes`** is one plot inside that figure — its own coordinate system, ticks, title, legend.

A figure can hold one axes (most plots) or a grid of many (a dashboard). You always create them with `fig, ax = plt.subplots()` and then draw onto `ax`. That is the **object-oriented interface**, and it is what you should use.

The alternative is the pyplot global interface — `plt.plot(...)` without an `ax` — which writes onto the "current" axes implicitly. That is the same kind of mistake as relying on `System.out` everywhere instead of passing a writer. Convenient at the REPL, fragile in any real script. **Always use the explicit form.**


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# The canonical pattern: create a figure with one axes
fig, ax = plt.subplots(figsize=(6, 3.5))

x = np.linspace(0, 10, 50)
ax.plot(x, np.sin(x))

plt.show()
print("type of fig:", type(fig).__name__)
print("type of ax: ", type(ax).__name__)


## Anatomy of an Axes

Every plot has the same structural pieces, all set via methods on the `Axes` object.

| Method | What it sets |
|---|---|
| `ax.set_title(...)` | The plot title |
| `ax.set_xlabel(...)` / `ax.set_ylabel(...)` | Axis labels |
| `ax.set_xlim(...)` / `ax.set_ylim(...)` | Axis ranges |
| `ax.legend(...)` | Show the legend |
| `ax.grid(True)` | Toggle gridlines |
| `ax.set_xticks(...)` | Tick positions |

The convention is to draw your data first, then set everything else. To put multiple series on the same axes, call `ax.plot(...)` (or any other plot method) twice. Both lines share the same axes; matplotlib gives them different colors automatically and includes them in the legend when you label them.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

x = np.linspace(0, 10, 50)
ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)")

ax.set_title("Trig functions over [0, 10]")
ax.set_xlabel("x")
ax.set_ylabel("value")
ax.legend()

plt.show()


## Subplots — a grid of axes

`plt.subplots(nrows, ncols)` returns a `Figure` and a 2D array of `Axes`. Each axes is independent — its own data, its own title, its own ticks. You index into the array like any NumPy array.

The pattern for a dashboard is short. Create the grid, draw onto each cell, finish with `fig.tight_layout()`.

`fig.tight_layout()` is the one-liner that prevents subplot titles and axis labels from overlapping. Run it on every multi-axes figure.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
rng = np.random.default_rng(42)

x = np.linspace(0, 10, 100)
axes[0, 0].plot(x, np.sin(x))
axes[0, 0].set_title("Line plot")

axes[0, 1].scatter(rng.normal(size=50), rng.normal(size=50))
axes[0, 1].set_title("Scatter plot")

axes[1, 0].hist(rng.normal(loc=50, scale=15, size=500), bins=20)
axes[1, 0].set_title("Histogram")

cities = ["Bengaluru", "Mumbai", "Pune", "Chennai"]
axes[1, 1].bar(cities, [4.2, 3.8, 2.1, 1.9])
axes[1, 1].set_title("Bar chart")

fig.tight_layout()
plt.show()


## Seaborn — statistical plots that take a DataFrame

The seaborn API is consistent across plot types. Every function accepts a long-form DataFrame plus column names as strings.

- `sns.histplot(data=df, x="column")` — distribution of one numeric column.
- `sns.boxplot(data=df, x="category", y="numeric")` — distribution split by category.
- `sns.scatterplot(data=df, x="col1", y="col2", hue="category")` — relationship between two numeric columns.
- `sns.barplot(data=df, x="category", y="numeric")` — mean of a numeric column per category.

The `hue=` argument is the single most important option to learn — it splits the plot by a categorical column and color-codes each group. One line, color-coded comparison.

Pass `ax=...` to drop a seaborn plot into a specific matplotlib axes. That is how you build dashboards with seaborn parts.


In [ ]:
rng = np.random.default_rng(42)

customers = pd.DataFrame({
    "customer_id": range(1001, 1101),
    "age": rng.integers(22, 65, 100),
    "city": rng.choice(["Bengaluru", "Mumbai", "Hyderabad", "Chennai", "Pune"], 100),
    "segment": rng.choice(["retail", "wealth"], 100, p=[0.75, 0.25]),
    "monthly_income": rng.lognormal(mean=11, sigma=0.6, size=100).astype(int),
})

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(data=customers, x="monthly_income", bins=20, ax=ax)
ax.set_title("Distribution of monthly income across customers")
plt.show()


In [ ]:
# Distribution split by category — boxplot summarises each group's quartiles + outliers
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=customers, x="city", y="monthly_income", ax=ax)
ax.set_title("Monthly income by city")
plt.show()


## Relationship plots — `scatterplot`

When both axes are numeric, you want a scatterplot. `sns.scatterplot` is the workhorse. Adding `hue=` colors the points by a third (categorical) column. Adding `size=` scales the marker by a fourth (numeric) column.

For a quick linear-regression overlay, `sns.regplot` draws the scatter plus a fitted line with a confidence band. Useful for *"is there a relationship here?"* at a glance, before any formal correlation testing in notebook 06.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(
    data=customers,
    x="age",
    y="monthly_income",
    hue="segment",
    ax=ax,
)
ax.set_title("Age vs monthly income, split by customer segment")
plt.show()


## Categorical aggregates — `barplot`

`sns.barplot` takes long-form data and aggregates **for you** — by default it shows the mean of `y` for each `x` group, with a confidence interval bar on top.

This is different from `ax.bar(...)` in matplotlib, which expects you to have already aggregated. `sns.barplot` does the `groupby` internally. Add `hue=` to split bars by a second category — side-by-side bars per group, color-coded.

For categorical *counts* (not aggregates), use `sns.countplot`. Same idea but counts rows per category.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=customers,
    x="city",
    y="monthly_income",
    hue="segment",
    estimator="mean",
    errorbar=None,   # turn off the confidence-interval whiskers for a clean read
    ax=ax,
)
ax.set_title("Average monthly income by city, split by segment")
ax.set_ylabel("Average monthly income (INR)")
plt.show()


## Correlation heatmap

When you have several numeric columns and want to see all pairwise relationships at once, the move is `df.corr()` followed by `sns.heatmap`. The result is a colored square matrix; you can read off which columns track each other.

Two conventions worth following.

- Pass `annot=True` to print the correlation value in each cell — readable at a glance.
- Use a diverging colormap (`"coolwarm"`, `"RdBu_r"`) centered at zero, so positive and negative correlations are visually distinct.

A correlation of `+1` means the two columns move together exactly; `-1` means they move in opposite directions exactly; `0` means no linear relationship. The heatmap is the fastest way to spot redundant features before modeling — anything above `0.9` in absolute value is usually a duplicate signal.


In [ ]:
# Add a couple of numeric columns so the correlation picture is richer
customers["account_balance"] = (customers["monthly_income"] * rng.uniform(2, 24, len(customers))).astype(int)
customers["credit_score"] = (
    400 + (customers["monthly_income"] / 1000).clip(0, 200)
    + rng.normal(0, 50, len(customers))
).clip(300, 850).astype(int)

numeric_cols = ["age", "monthly_income", "account_balance", "credit_score"]
corr = customers[numeric_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix — Fintech Bank customers")
plt.show()


## What's next

You can now build any plot an EDA session needs. The figure/axes model gives you the structure; seaborn gives you the statistical plot types — distribution, relationship, categorical, correlation.

Notebook 06 introduces **statistics for data scientists** — descriptive stats you have already touched, plus distributions, hypothesis testing, and correlation as a *formal* measure. The heatmap above shows you the numbers; notebook 06 tells you when those numbers are meaningful.

Two from-memory exercises.

1. Build a small `card_transactions` DataFrame with `amount` and `category` columns. In a single figure with two axes side by side, plot a histogram of `amount` on the left and a boxplot of `amount` grouped by `category` on the right.
2. From the `customers` DataFrame in this notebook, produce a `scatterplot` of `monthly_income` vs `credit_score` with the points colored by `city`. Does any city stand out?
